# 02 — Understand the Operational Model

**Question this notebook answers:** How does a case actually move through the current MyLA311 system — what do `Status`, `ResolutionCode`, `Owner`, and the other operational fields mean?

Conclusions and the full lifecycle write-up live in [`docs/datasets/myla311_operational_model.md`](../docs/datasets/myla311_operational_model.md). This notebook is the evidence.

**On documentation:** the Socrata metadata carries official descriptions for some columns (`Owner`, `RequestSource`, `AssignTo`, `ServiceDate`…) but **none at all for `Status`, `ResolutionCode`, `ReasonCode`, or `ActionTaken`** — and the dataset has no attached data dictionary. For those four fields, everything below is *inference from the data itself*, cross-checked against official City program documentation where it exists (LASAN, OCB).

In [1]:
from pathlib import Path

import pandas as pd

df = pd.read_csv(Path("../data/raw/myla311_cases_2026.csv"), low_memory=False)
print(f"{len(df):,} cases")

1,209,648 cases


## Status — 9 values, small and learnable

Two techniques used throughout this notebook, both pandas workhorses:

- `groupby(col)[other].agg(...)` — split rows into groups by one column, compute something per group. The SQL equivalent of `GROUP BY`.
- The *boolean-mean trick*: `df["x"].notna()` gives True/False per row, and the mean of booleans is the *fraction that are True*. Grouping that by another column gives "% populated per group" in one line.

In [2]:
status = df.groupby("Status").agg(
    n=("CaseNumber", "size"),
    pct_has_closeddate=("ClosedDate", lambda s: round(s.notna().mean() * 100)),
)
status.sort_values("n", ascending=False)

,n,pct_has_closeddate
Status,,
Closed,1019428,100
Reported,65068,100
Workorder Created,51665,0
Cancelled,42755,100
New,14729,0
In Progress,7100,0
Closed Ext-Referred,5914,100
Potential Duplicate,2576,0
Duplicate Confirm,413,0


Reading this: `Closed`, `Cancelled`, `Closed Ext-Referred` — and, surprisingly, **`Reported`** — always carry a `ClosedDate`; `New`, `Workorder Created`, `In Progress`, and the duplicate states never do. So the state machine splits into *live* states and *terminal* states, and `Reported` behaves like a terminal state. Which request types end up `Reported`?

In [3]:
df[df["Status"] == "Reported"]["RequestType"].value_counts().head(8)

RequestType
Homeless Encampment              52626
Private Property Violation        5797
Report a Sidewalk Problem         5565
Report Water Wasting               559
Curb Issues                        414
Street Pavement Issues              62
Dockless Mobility Enforcement       43
Other                                1
Name: count, dtype: int64

`Reported` is the terminal state for **report-only case types** — homeless encampments, private property violations, sidewalk problems. The city acknowledges the report (`ResolutionCode` = *"C-Report has been received"*) but does not open an individual work order through 311. Official context: encampment reports feed LASAN's Livability Services Division (LSD), whose CARE/CARE+ teams work off **scheduled regional deployments**, not one work order per 311 report.

## ResolutionCode — 141 values, department-specific vocabularies

No official documentation. But the prefixes and content make the structure clear: each department brought its own closure vocabulary.

In [4]:
print(f'{df["ResolutionCode"].nunique()} unique values\n')
df["ResolutionCode"].value_counts().head(25)

141 unique values



ResolutionCode
AR-Request Completed                                385993
C-Closed                                            325169
QC-Item Not Out                                     118396
C-Report has been received                           58648
DUP-Duplicate                                        44170
RC-Contractor Serviced                               35056
1004-Closed                                          29398
RCAN-Cancelled by Contractor                         16620
SARH-SAR completed-Hot                               12962
RC-Request Completed                                  8216
1005-Cancelled                                        7944
GI-General Information                                7601
COM-Request Completed                                 6885
1003-Referral to External Department                  6197
VMRB-Vehicle moved for rebalancing                    5541
DUP-Duplicate SR                                      4646
B-Duplicated Request                     

The critical finding: **`Closed` does not mean the problem was fixed.** The codes separate real outcomes:

- *Work done:* `AR-Request Completed`, `RC-Contractor Serviced`, `SARH-SAR completed-Hot` (StreetsLA **S**mall **A**sphalt **R**epair, hot-mix), `PFR-Palm Fronds Removed`, `TR-Tree Removed`
- *Crew came, nothing to do:* `QC-Item Not Out` (**118K cases** — third most common outcome!), `NCPP-Not at Curb / On Private Property`, `ASG-Already Serviced`, `NF` in ReasonCode
- *Administrative closure:* `C-Closed`, `1004-Closed`, `RF-Receive and File`, `NAT-No Action Taken`, `GI-General Information`
- *Duplicates/cancellations:* `DUP-*`, `CDR-*`, `RCAN-Cancelled by Contractor`, `1005-Cancelled`
- *Referrals:* `1003-Referral to External Department`

Which types produce `QC-Item Not Out`?

In [5]:
df[df["ResolutionCode"] == "QC-Item Not Out"]["RequestType"].value_counts().head(4)

RequestType
Illegal Dumping Item Pickup    60388
Item Pickups                   51633
Dead Animal Removal             4149
Service Not Complete            2216
Name: count, dtype: int64

## ReasonCode — sparse, mixed, department-specific

91% empty. The populated values mix inspection outcomes (`I-Inspected`), scheduling steps (`SARS-Small Asphalt Repair (SAR) scheduled`), failure explanations (`0200-Blocked in By Vehicle/Object`), and case-hygiene codes (`0999-Wrong SR Type`). It reads as an optional per-department workflow annotation, not a systematic field.

In [6]:
df["ReasonCode"].value_counts().head(20)

ReasonCode
I-Inspected                                  42955
SARS-Small Asphalt Repair (SAR) scheduled    16873
0200-Blocked in By Vehicle/Object             8876
0240-SR-Scheduled                             4765
RPF-Remove Palm Fronds                        3496
OFH-Off of hold                               3041
0999-Wrong SR Type                            2990
NF-Nothing found                              2266
0222-Ahead of Regular Service                 1860
RL-Remove Limb                                1758
0206-Same Day Collection                      1699
NOV-Notice of Violation issued                1664
1000-Change Internal Department               1660
0237-SR Follow-Up                             1438
IV-Investigate                                1229
0214-Moved/Missing                             924
RT-Remove Tree                                 906
0212-Not SNC - On Call Request                 810
0225-Reschedule Container Not Out              666
0001-Request Receive

## ActionTaken — the call-center agent's log

Officially undocumented, but the data tells a crisp story: check how often it's populated per intake channel.

In [7]:
(df.assign(populated=df["ActionTaken"].notna())
   .groupby("RequestSource")["populated"].mean().round(2)
   .sort_values(ascending=False))

RequestSource
Call                        1.00
Social                      1.00
City Attorney               1.00
Web Form                    1.00
Walk-in                     1.00
Letter                      1.00
Chat                        1.00
Radio                       1.00
Email                       0.99
Council's Office            0.98
Voicemail                   0.98
Driver Self Report          0.96
Mayor's Office              0.50
Fax                         0.50
Self Service                0.00
Mobile App                  0.00
Driver Self Reported        0.00
recycLA Service Provider    0.00
Name: populated, dtype: float64

In [8]:
df["ActionTaken"].value_counts()

ActionTaken
SR Created                  335915
Information Provided        205070
Transferred                  43314
Status Provided              21942
Unable to Assist              1041
SR Update                      802
Referred to Other Agency       648
Escalate to Supervisor         286
Consultation/3-way              25
Radio Dispatch                   3
Name: count, dtype: int64

~100% populated for **agent-handled channels** (Call, Email, Chat, Voicemail, Council's Office), ~0% for **self-service channels** (web portal, Mobile App, recycLA). `ActionTaken` records what the human agent did: created a service request, provided information, transferred the call, gave a status update. It is an *intake* field, not a field-work field.

## Owner — who the case is routed to

Officially: *"City agency or department assigned to this ticket."* 96 distinct values, dominated by a few: 

In [9]:
print(f'{df["Owner"].nunique()} unique values\n')
df["Owner"].value_counts().head(20)

99 unique values



Owner
LASAN                                    617700
ITA                                      164613
OCB - Normal Cases                        72780
LSD                                       49645
LASAN - LSD                               49381
RAP - Construction                        46172
BSS - SMD                                 44016
BSL                                       25216
recycLA                                   25181
BSS - IED                                 24019
LADOT - District Operations               18984
BSS - UFD                                 15324
LADOT - Dockless Mobility Enforcement     11386
LASAN - CCD Staff                          9366
DPW                                        6742
LADBS                                      5801
LAAS                                       4325
RAP - Homeless Related Services Team       2985
BSS - UTA                                  2799
OCB - Special Cases                        2449
Name: count, dtype: int64

In [10]:
for rt in df["RequestType"].value_counts().head(8).index:
    top = df[df["RequestType"] == rt]["Owner"].value_counts().head(3)
    print(f"{rt}: " + " | ".join(f"{k} ({v:,})" for k, v in top.items()))

Information-Only: ITA (161,267) | LASAN (119,067) | OCB (1)


Item Pickups: LASAN (241,354) | LASAN - LSD (3,701) | recycLA (716)
Illegal Dumping Item Pickup: LASAN (198,210) | LASAN - LSD (44,125) | Street Services Bureau (Public Works) (131)


Graffiti Removal: OCB - Normal Cases (72,773) | RAP - Construction (45,556) | OCB - Special Cases (2,448)


Service Not Complete: LASAN (41,717) | recycLA (24,465)
Homeless Encampment: LSD (49,641) | RAP - Homeless Related Services Team (2,985)
Street Pavement Issues: BSS - SMD (25,794) | DPW (756) | LADOT - District Operations (754)
Streetlight Repair Services: BSL (24,576) | Department of Water and Power (9) | Recreation and Parks (3)


Routing is clean at the top: LASAN owns collections/dumping, ITA (which runs the 311 center) owns `Information-Only`, OCB owns graffiti, BSL owns streetlights, `LSD` (LASAN's Livability Services Division) owns encampments. Anomalies: `LSD` vs `LASAN - LSD` are the same unit under two labels; `RAP - Construction` owning ~45K graffiti cases is unexplained; the tail includes staff names and a literal `TEST XX`.

## AssignTo — the crew/contractor level below Owner

Officially: *"The specific group that is assigned to this request."*

In [11]:
print(f'{df["AssignTo"].nunique()} unique values\n')
df["AssignTo"].value_counts().head(20)

49 unique values



AssignTo
NC           124954
EV           120430
SLA          108743
WV            91538
WLA           83578
CCAC          53135
BSS - SMD     43486
HB            29538
BSS - IED     24012
BSS - UFD     15322
WVA           12676
KYCC           7961
Lime           7370
GAPBH          6629
LA Corps       6181
NDFY           5461
HBT            4537
CRCD           4430
PGS            3962
NEGB           3760
Name: count, dtype: int64

In [12]:
df[df["RequestType"] == "Graffiti Removal"]["AssignTo"].value_counts().head(10)

AssignTo
CCAC        53135
WVA         12676
KYCC         7961
GAPBH        6629
LA Corps     6181
NDFY         5461
HBT          4537
CRCD         4430
PGS          3962
NEGB         3760
Name: count, dtype: int64

Two populations: **LASAN service-yard districts** (EV = East Valley, WV = West Valley, WLA, SLA = South LA, NC = North Central, HB = Harbor) and **named contractors**. The graffiti breakdown is the smoking gun: CCAC (Central City Action Committee), KYCC (Koreatown Youth & Community Center), NEGB (Northeast Graffiti Busters), HBT (Hollywood Beautification Team), LA Corps, CRCD, etc. — these are the community-based organizations OCB contracts for graffiti abatement, with a published 72-hour removal goal. `Lime` appears for dockless scooter cases. **The city's maintenance workforce is substantially contracted out, and this field exposes it.**

## RequestSource — intake channels

In [13]:
df["RequestSource"].value_counts()

RequestSource
Call                        599584
Self Service                579384
recycLA Service Provider     12478
Driver Self Reported          6200
Mobile App                    3673
Council's Office              3491
Email                         3100
Voicemail                     1023
Chat                           351
Social                         122
Driver Self Report              28
Web Form                        24
Letter                          23
City Attorney                   20
Walk-in                          6
Fax                              2
Mayor's Office                   2
Radio                            1
Name: count, dtype: int64

Call (49.6%) and Self Service (47.9%) dwarf everything. Note the *non-citizen* channels: `recycLA Service Provider` (franchise waste haulers filing cases, mostly `Service Not Complete`) and `Driver Self Report(ed)` (city crews logging work they found in the field — in two spellings). `Mobile App` at only 3.7K is startling — the old system's app was a major channel; either usage collapsed or app submissions now flow in as `Self Service`.

## The `Service Not Complete` type — resolved from Checkpoint 2

In [14]:
snc = df[df["RequestType"] == "Service Not Complete"]
print("Owner:", snc["Owner"].value_counts().head(3).to_dict())
print("Source:", snc["RequestSource"].value_counts().head(3).to_dict())
print("Resolution:", snc["ResolutionCode"].value_counts().head(4).to_dict())

Owner: {'LASAN': 41717, 'recycLA': 24465}


Source: {'Call': 45745, 'recycLA Service Provider': 12376, 'Self Service': 7893}
Resolution: {'AR-Request Completed': 33279, 'RCAN-Cancelled by Contractor': 16473, 'COM-Request Completed': 6557, 'ASG-Already Serviced': 2459}


It is **not** a general recurrence signal: it's a trash-collection workflow. Owners are LASAN and recycLA; a quarter are filed *by the waste haulers themselves*; resolutions are "completed" or "cancelled by contractor." It flags missed collections, not unresolved civic problems.

## Duplicate handling and instant closes

In [15]:
io = df[df["RequestType"] == "Information-Only"]
created = pd.to_datetime(io["CreatedDate"], format="mixed")
closed = pd.to_datetime(io["ClosedDate"], format="mixed")
frac = ((closed - created).dt.total_seconds() <= 60).mean()
print(f"Information-Only closed within 60s of creation: {frac:.1%}")

print("\nPotential Duplicate:", df[df['Status'] == 'Potential Duplicate']['RequestType'].value_counts().head(3).to_dict())
print("Duplicate Confirm:  ", df[df['Status'] == 'Duplicate Confirm']['RequestType'].value_counts().head(3).to_dict())

Information-Only closed within 60s of creation: 100.0%

Potential Duplicate: {'Graffiti Removal': 797, 'Traffic Safety': 648, 'Curb Issues': 427}
Duplicate Confirm:   {'Curb Issues': 138, 'Traffic Safety': 123, 'Parking Issues': 40}


`Information-Only` cases are created and closed in the same breath — they're call log entries, not work. The duplicate states are a two-step review flow (`Potential Duplicate` → confirmed or returned to service), tiny in volume (~3K).

**Everything above is a snapshot, not a history.** Each case appears once, in its *current* state — we cannot see state transitions, timestamps per state, or reopenings. The lifecycle in the companion doc is reconstructed logic, not observed sequence. See [`docs/datasets/myla311_operational_model.md`](../docs/datasets/myla311_operational_model.md) for the assembled model.